# In the paper ["Harsh", "Balanced", "Lenient"] == ["Bad", "Middle", "Good"] here.

## We perform prompt optimisation with:
## 1. Expert Annotations + Score from experts
## 2. Score-only from experts

## No_experts stands for score-only signal.

### Basically all annotated training bundles comes with an expert annotations and a score out of 5. No_experts stands for only using the score/5 to guide prompt optimisation which is misleading but I'm too lazy to change it.

In [4]:
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git && cd LLM4BEAR && git sparse-checkout set 1_EGPO

Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 7 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (7/7), done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 67 bytes | 67.00 KiB/s, done.
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 10 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), 9.86 KiB | 9.86 MiB/s, done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


from google.colab import userdata
my_secret_key = userdata.get('API_KEY')

wandb_secret_key = userdata.get('WANDB_KEY')

if my_secret_key:
  print("Token retrieved successfully.")
else:
  print("Token not found in Colab Secrets.")

from openai import OpenAI

client = OpenAI(
    # This is the default and can be omitted
    api_key = my_secret_key,
)


import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key = my_secret_key,
)  # make sure this is your actual key


In [5]:
%cd LLM4BEAR/1_EGPO/

!pip install -r requirements.txt

import random
import wandb
import json
import random
import pickle
import numpy as np

from tqdm import tqdm
from opt.config import init_config
from opt.request import Request
from opt.reward import Reward
from opt.improve import Improve
from opt.select import Select

/content/LLM4BEAR/1_EGPO/LLM4BEAR/1_EGPO
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [6]:
import os

path = "/content/drive/MyDrive/EGPO/final_prompts/"

try:
    os.makedirs(path, exist_ok=True)
    print(f"Successfully created: {path}")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully created: /content/drive/MyDrive/EGPO/final_prompts/


In [ ]:

async def get_top_3_prompts(beam_candidate, val_data, reward_model, result_table):
    """
    Calculates rewards for each prompt in the beam_candidate list
    and returns the 5 prompts with the highest rewards.
    """
    sample_data = list(val_data.values())
    reward_prompt_pairs = []

    # 1. Calculate the reward for each prompt and pair them up
    for prompt in beam_candidate:
        reward = await reward_model.calculate_reward(prompt, sample_data)

        print(reward)
        reward_prompt_pairs.append((reward, prompt))
        if result_table is not None:
            result_table.add_data(prompt, reward)

    # 2. Sort the list of (reward, prompt) tuples in descending order by reward
    sorted_pairs = sorted(reward_prompt_pairs, key=lambda item: item[0], reverse=True)

    # 3. Extract just the prompts from the top 3 pairs
    top_3_prompts = [prompt for reward, prompt in sorted_pairs[:3]]

    # filename = f"/content/drive/MyDrive/EGPO/final_prompts/expert_refined_top_3.pkl" # we save in a different function but you can save here for testing purposes i guess.

    # with open(filename, 'wb') as f:
    #     pickle.dump(top_3_prompts, f)

    return top_3_prompts


/content/LLM4BEAR/1_EGPO


In [ ]:

async def structure_refinement(char, initial_prompt, inferring_reasons, refining_prompts, augmenting_prompts, training_data, validation_data, constant_metrics = "", metastructure=None, mode = 'single', consideration="", convincing=""):

    god_given_criteria =  "You are an expert bundle evaluator. Use this criteria to help you evaluate the bundle:\n"\
                          "- 1 - Poor: Items do not correlate with each other or the intent.\n"\
                          "- 2 - Needs Improvement: Some items are connected, but multiple modifications are needed to ensure the bundle's acceptability.\n"\
                          "- 3 - Almost: Only one modification is needed to guarantee the acceptability of the bundle.\n"\
                          "- 4 - Acceptable: Not perfect, but the intended user would find the bundle appealing.\n"\
                          "- 5 - Excellent: No flaws with the bundle.\n"

    if consideration == "":
        json_addition = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
                        "**JSON Schema:**\n"\
                        "```json\n"\
                        "{{\n"\
                        "score: float, bundle score out of 5 given to 2 decimal places\n"\
                        "}}"

    elif consideration == "1-2":
        json_addition = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
                        "**JSON Schema:**\n"\
                        "```json\n"\
                        "{{\n"\
                        "score: float, bundle quality out of 5.\n"\
                        "is_poor_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
                        "is_acceptable_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
                        "}}"

    elif consideration == "3":
        json_addition = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
                        "**JSON Schema:**\n"\
                        "```json\n"\
                        "{{\n"\
                        "score: float, bundle quality out of 5.\n"\
                        "needs_improvement_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
                        "is_good_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
                        "score: float, bundle quality out of 5.\n"\
                        "}}"


    elif consideration == "4-5":

        json_addition = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
                        "**JSON Schema:**\n"\
                        "```json\n"\
                        "{{\n"\
                        "score: float, bundle quality out of 5.\n"\
                        "needs_improvement_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
                        "is_high_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
                        "}}"



    else:
        json_addition = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
                        "**JSON Schema:**\n"\
                        "```json\n"\
                        "{{\n"\
                        "score: float, bundle quality out of 5.\n"\
                        "verdict: str, yes/no response, do not provide anything other than yes or no.\n"\
                        "}}"


    print(initial_prompt)
    print(metastructure)

    conf = init_config()
    conf['initial_prompt'] = initial_prompt
    conf['inferring_reasons'] = inferring_reasons
    conf['refining_prompts'] = refining_prompts
    conf['augmenting_prompts'] = augmenting_prompts
    conf['json_addition'] = json_addition
    conf['metrics'] = constant_metrics
    conf['opt_type'] = mode
    conf['base_reward'] = 3
    conf['threshold'] = 0.95 # this is a legacy parameter, before we used an ensemble strategy and tried to optimise for a single perfect prompt. That is when consideration == "", which is commented out.
    conf['case'] = consideration
    conf['convince'] = convincing

    conf['initialise_struct'] = "You are an expert in summarising thorough analyses. Please use the given template to organise the given analysis.\n"
    conf['initialise_judge'] = god_given_criteria
    conf['operation'] = """These are the operations:
Add: introduce another item to the bundle.
Remove: remove an item from the bundle.
Replace: replace an item in the bundle with another item.
Decompose: split the bundle into 2 or more bundles.
"""


    # conf['epsilon'] = epsilon

    conf['wandb_api_key'] = wandb_secret_key
    conf['openai_api_key'] = my_secret_key

    opt_request = Request(conf)

    if conf['use_wandb']:
        wandb.login(key=conf['wandb_api_key'])
        # conf.pop('openai_api_key')
        run = wandb.init(
            project=f"PO4ISR_{conf['dataset']}_tune",
            config=conf,
        )
        text_table = wandb.Table(columns=["Input", "Prompt", "Reason", "Improved prompt", "Augumented prompt"])
        reward_table = wandb.Table(columns=["Prompt", "Reward"])
    else:
        text_table = None
    print("parameter initialization is complete")


    train_data = training_data
    val_data = validation_data

    val_data_values = list(val_data.values())
    val_input = [data['input'] for data in val_data_values]
    val_scores = [data['target_score'] for data in val_data_values]

    prompt_data = [{"prompts": data + "\n" + json_addition} for data in val_input]

    beam_candidate = []

    random.seed(conf['seed'])


    opt_reward = Reward(conf, opt_request)
    opt_improve = Improve(inferring_reasons, refining_prompts, augmenting_prompts, train_data, conf, opt_request)
    opt_select = Select(train_data, conf, opt_reward)

    print()
    print()
    print("==============")
    print("The apo algorithm is running...")
    print("==============")

    if conf['opt_type'] == 'single':
        beam_candidate.append(initial_prompt)
    elif conf['opt_type'] == 'structure':
        beam_candidate.append(metastructure)

    all_prompts = []

    runnings = 0

    keep_going = True


    while keep_going:
        # print()
        # print()
        print("Search depth: " + str(int(runnings+1)))


        all_expanded_candidates = []
        beam_count = 1

        if conf['opt_type'] == 'single':

            for prompt in beam_candidate:
                print()
                print(f"beam_count: {beam_count}/{len(beam_candidate)}")
                print()
                # Expand
                expanded_prompts = await opt_improve.run(prompt, table=text_table)

                all_expanded_candidates.extend(expanded_prompts)
                beam_count += 1

        # Select
        if conf['opt_type'] == 'single':

            beam_candidate, top_1_prompt = await opt_select.run(all_expanded_candidates) # top_1_prompt is left there as legacy code, if you want to take a single best performing candidate you can.


        all_prompts.append(beam_candidate)

        # Argmax prompt
        # print("Trying to find top-1 prompt\n")


        if runnings < 4:
            conf['threshold'] += -0.1

        runnings += 1

        # Add a safeguard against infinite loops
        if runnings == conf['search_depth']:
            keep_going = False

    try:
        last_2_iterations = [item for sublist in all_prompts[-4:] for item in sublist]
    except:
        last_2_iterations = all_prompts[-1]

    if conf['opt_type'] == 'single':
        top_3 = await get_top_3_prompts(last_2_iterations, val_data, opt_reward, reward_table)

    filename = f"/content/drive/MyDrive/EGPO/final_prompts/Refined_{char}.pkl"



    with open(filename, 'wb') as f:
        pickle.dump([top_3, all_prompts], f)

    return

In [ ]:
adding_metrics = "\nFunctionality Integration: Describe how a user would utilize this collection of items to achieve their primary goal. Considering the entire workflow, is this a complete and logical set of items for the task, or is there an irrelevant or missing item? \n"\
              "Similarity: What is the common theme or category that connects these items?\n"\
              "Complementarity: Are these items more valuable together than they would be if sold separately? Does the presence of one item create a clear reason to buy the other(s)?\n"\
              "Diversity: Does the variety of items in this bundle cater to a broad set of related needs for a single user, or does the mix of items seem unfocused and random?\n"


with open(f"./Dataset/bundle/Text/electronic_train_50.json", 'r') as json_file:
    electronic_train_data = json.load(json_file)
with open(f"./Dataset/bundle/Text/valid.json", 'r') as json_file:
    electronic_val_data = json.load(json_file)


with open(f"./Dataset/bundle/Text/clothing_train_50.json", 'r') as json_file:
    clothing_train_data = json.load(json_file)
with open(f"./Dataset/bundle/Text/clothing_valid.json", 'r') as json_file:
    clothing_val_data = json.load(json_file)


with open(f"./Dataset/bundle/Text/food_train_50.json", 'r') as json_file:
    food_train_data = json.load(json_file)
with open(f"./Dataset/bundle/Text/food_valid.json", 'r') as json_file:
    food_val_data = json.load(json_file)


In [ ]:
from typing_extensions import get_overloads


# initial_prompt = "You are an expert bundle strategist tasked distinguishing a $quality$/5 quality bundle. Your response should be a single judgement, is this bundle a $quality$/5 bundle?: yes/no. \n"\
#                   "Your task is to diligently complete subtasks that can help guide your analysis step by step:\n" \
#                   "1. Based on the stated intent, come up with how combinations of items can interact to fulfill the intent.\n" \
#                   "2. If from your understanding, the combinations do not meet the stated intent, develop a new intent that your combinations can fulfill.\n" \
#                   "3. Based on your reasoning and analysis, you are to make two mutually exclusive yes/no judgements on the bundle. $criteria$" \
#                   "4. Now, you are to analyse the bundle: \n"

# Importance graph reasoning generally allows better prompts to be refined.

initial_importance_prompt = "You are an expert bundle strategist tasked distinguishing a $quality$/5 quality bundle. Your response should be a single judgement, is this bundle a $quality$/5 bundle?: yes/no. \n"\
                  "Your task is to diligently complete subtasks that can help guide your analysis step by step:\n" \
                  "1. Based on the stated intent, come up with how combinations of items can interact to fulfill the intent.\n" \
                  "2. If from your understanding, the combinations do not meet the stated intent, develop a new intent that your combinations can fulfill.\n" \
                  "3. Design a importance graph analysis for each bundle in the format:\n"\
                  "a. [Most Important Bundle Item] — [Role: Primary/Secondary/Tertiary]\n"\
                  "Reason: [Your explanation for this item's rank in this specific scenario]\n"\
                  "b. [2nd Most Important Bundle Item] — [Role: Primary/Secondary/Tertiary]\n"\
                  "Reason: [Your explanation]\n"\
                  "(...and so on for all other items)\n"\
                  "4. Based on your reasoning and analysis, you are to make two mutually exclusive yes/no judgements on the bundle. $criteria$" \
                  "5. Now, you are to analyse the bundle: \n"

bad_bundle_criteria = "\nFollow this criteria when evaluating the bundle:\n"\
                "1-2 - Poor: Some items or no items are connected thematically, but more than one modification needs to be made to guarantee the acceptability of the bundle.\n"\
                "3-5 - Acceptable: Items are connected thematically, and complement each other.\n"

middle_bundle_criteria = "\nFollow this criteria when evaluating the bundle:\n"\
                "1-3 - Needs Improvement: One or more modifications needs to be made in order to guarantee the acceptability of the bundle.\n"\
                "4-5 - Good Quality: No modifications need to be performed as this bundle is extremely well designed.\n"

good_bundle_criteria = "\nFollow this criteria when evaluating the bundle:\n"\
                "1-3 - Needs Improvement: One or more modifications needs to be made in order to guarantee the acceptability of the bundle.\n"\
                "4-5 - High Quality: No modifications need to be performed to be accepted by the end user.\n"


# initial_prompt_bad_bundle = initial_prompt.replace("$criteria$",
#                                                          bad_bundle_criteria).replace("$quality$", "1-2")

# initial_prompt_middle_bundle = initial_prompt.replace("$criteria$",
#                                                          middle_bundle_criteria).replace("$quality$", "1-3")

# initial_prompt_good_bundle = initial_prompt.replace("$criteria$",
#                                                          good_bundle_criteria).replace("$quality$", "4-5")

initial_importance_prompt_bad_bundle = initial_importance_prompt.replace("$criteria$",
                                                         bad_bundle_criteria).replace("$quality$", "1-2")

initial_importance_prompt_middle_bundle = initial_importance_prompt.replace("$criteria$",
                                                         middle_bundle_criteria).replace("$quality$", "1-3")

initial_importance_prompt_good_bundle = initial_importance_prompt.replace("$criteria$",
                                                         good_bundle_criteria).replace("$quality$", "4-5")

In [ ]:
convincing_prompt = """You are a senior analyst and adjudicator. Your job is to review a disagreement between a junior AI analyst and a human expert to determine if the expert's reasoning is sound and should be used as a training example.

--- CONTEXT ---

1.  **Bundle in Question:**
    $error_case$

2.  **Disagreement Summary:**
    $llm_judge$

3.  **Junior AI's Reasoning:**
    $llm_reasoning$

4.  **Expert's Score and Reasoning:**
    - Score: $true_score$/5
    - Reasoning: $annotation$

--- YOUR TASK ---

Perform your task in two parts.

**Part 1: Written Analysis**
First, write a brief analysis. Is the expert's argument convincing, logical, and specific? Treat the expert's reasoning as more accurate AI's reasoning unless it seems that expert overlooked a detail that the AI considered?
Second, if the expert seems correct, then you must identify what is the most important point to the expert, and what was most important to the AI which led to a disagreement.

**Part 2: Final Verdict (JSON)**
After your analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object with your final verdict. The verdict must be either "yes" (the expert is convincing) or "no".

**JSON Schema:**
```json
{{
    "verdict": str, yes/no response, do not provide anything else.
}}
"""

inferring_reasons = "I'm trying to refine a prompt that evaluated the following bundle incorrectly: $error_case$.\n"\
                    "$llm_judge$ The reasoning given was: $llm_reasoning$ \n"\
                    "The expert gave the bundle a score of $true_score$, with the reasoning:\n $annotation$ \n"\
                    "First, write a brief analysis. Is the expert's argument convincing, logical, and specific? Treat the expert's reasoning as more accurate AI's reasoning unless it seems that expert overlooked a detail that the AI considered?\n"\
                    "Second, if the expert seems correct, then you must identify what is the most important point to the expert, and what was most important to the AI which led to a disagreement.\n"\
                    "Summarise the difference in reasoning between the LLM and the expert, and incorporate what specific criteria and information did the expert use that the LLM overlooked.\n"\
                    "Give $num_feedbacks$ reasons of why the LLM got this example wrong.\n"\
                    "Wrap each reason with <START> and <END>"

inferring_reasons_no_experts =  "I'm trying to refine a prompt that evaluated the following bundle incorrectly: $error_case$.\n"\
                                "$llm_judge$ The reasoning given was: $llm_reasoning$ \n"\
                                "Give $num_feedbacks$ reasons of why the LLM could have gotten this example wrong.\n"\
                                "Wrap each reason with <START> and <END>"

refining_prompts = "I'm trying to write a zero-shot evaluation prompt.\n"\
                    "My current prompt is \"$prompt$\"\n"\
                    "But this prompt gets the following example wrong: $error_case$\n"\
                    "$llm_judge$."\
                    "Based on these example the problem with this prompt is that $reasons$.\n"\
                    "Based on the above information, please write an improved prompt without including any bundle items.\n"\
                    "The prompt should be wrapped with <START> and <END>.\n"\
                    "The new prompt is:"

augmenting_prompts = "Generate a variation of the following instruction while maintaining the semantic meaning.\n"\
                      "Input: $refined_prompt$\n"\
                      "The prompt should be wrapped with <START> and <END>.\n"\
                      "Output:"

In [ ]:
# evaluator_configs = [
#     {
#         "char": "bad_evaluator_clothing_importance",
#         "initial_prompt": initial_importance_prompt_bad_bundle,
#         "consideration": "1-2"
#     },
#     {
#         "char": "middle_evaluator_clothing_importance",
#         "initial_prompt": initial_importance_prompt_middle_bundle,
#         "consideration": "3"
#     },
#     {
#         "char": "good_evaluator_clothing_importance",
#         "initial_prompt": initial_importance_prompt_good_bundle,
#         "consideration": "4-5"
#     }
# ]

# # Loop through the configurations to run the refinement process
# for config in evaluator_configs:
#     # Run the version with metrics
#     await structure_refinement(
#         char = config["char"],
#         initial_prompt = config["initial_prompt"],
#         inferring_reasons = inferring_reasons,
#         refining_prompts = refining_prompts,
#         augmenting_prompts = augmenting_prompts,
#         training_data = clothing_train_data,
#         validation_data = clothing_val_data,
#         constant_metrics = adding_metrics,
#         metastructure = None,
#         mode = 'single',
#         consideration = config["consideration"],
#         convincing = convincing_prompt
#     )




# evaluator_configs = [
#     {
#         "char": "bad_evaluator_electronic_importance",
#         "initial_prompt": initial_importance_prompt_bad_bundle,
#         "consideration": "1-2"
#     },
#     {
#         "char": "middle_evaluator_electronic_importance",
#         "initial_prompt": initial_importance_prompt_middle_bundle,
#         "consideration": "3"
#     },
#     {
#         "char": "good_evaluator_electronic_importance",
#         "initial_prompt": initial_importance_prompt_good_bundle,
#         "consideration": "4-5"
#     }
# ]

# # Loop through the configurations to run the refinement process
# for config in evaluator_configs:
#     # Run the version with metrics
#     await structure_refinement(
#         char = config["char"],
#         initial_prompt = config["initial_prompt"],
#         inferring_reasons = inferring_reasons,
#         refining_prompts = refining_prompts,
#         augmenting_prompts = augmenting_prompts,
#         training_data = electronic_train_data,
#         validation_data = electronic_val_data,
#         constant_metrics = adding_metrics,
#         metastructure = None,
#         mode = 'single',
#         consideration = config["consideration"],
#         convincing = convincing_prompt
#     )



# evaluator_configs = [
#     {
#         "char": "bad_evaluator_clothing_importance_no_experts",
#         "initial_prompt": initial_importance_prompt_bad_bundle,
#         "consideration": "1-2"
#     },
#     {
#         "char": "middle_evaluator_clothing_importance_no_experts",
#         "initial_prompt": initial_importance_prompt_middle_bundle,
#         "consideration": "3"
#     },
#     {
#         "char": "good_evaluator_clothing_importance_no_experts",
#         "initial_prompt": initial_importance_prompt_good_bundle,
#         "consideration": "4-5"
#     }
# ]

# # Loop through the configurations to run the refinement process
# for config in evaluator_configs:
#     # Run the version with metrics
#     await structure_refinement(
#         char = config["char"],
#         initial_prompt = config["initial_prompt"],
#         inferring_reasons = inferring_reasons_no_experts,
#         refining_prompts = refining_prompts,
#         augmenting_prompts = augmenting_prompts,
#         training_data = clothing_train_data,
#         validation_data = clothing_val_data,
#         constant_metrics = adding_metrics,
#         metastructure = None,
#         mode = 'single',
#         consideration = config["consideration"],
#         convincing = ""
#     )





# evaluator_configs = [
#     {
#         "char": "bad_evaluator_electronic_importance_no_experts",
#         "initial_prompt": initial_importance_prompt_bad_bundle,
#         "consideration": "1-2"
#     },
#     {
#         "char": "middle_evaluator_electronic_importance_no_experts",
#         "initial_prompt": initial_importance_prompt_middle_bundle,
#         "consideration": "3"
#     },
#     {
#         "char": "good_evaluator_electronic_importance_no_experts",
#         "initial_prompt": initial_importance_prompt_good_bundle,
#         "consideration": "4-5"
#     }
# ]

# # Loop through the configurations to run the refinement process
# for config in evaluator_configs:
#     # Run the version with metrics
#     await structure_refinement(
#         char = config["char"],
#         initial_prompt = config["initial_prompt"],
#         inferring_reasons = inferring_reasons_no_experts,
#         refining_prompts = refining_prompts,
#         augmenting_prompts = augmenting_prompts,
#         training_data = electronic_train_data,
#         validation_data = electronic_val_data,
#         constant_metrics = adding_metrics,
#         metastructure = None,
#         mode = 'single',
#         consideration = config["consideration"],
#         convincing = ""
#     )




# evaluator_configs = [
#     {
#         "char": "bad_evaluator_food_importance",
#         "initial_prompt": initial_importance_prompt_bad_bundle,
#         "consideration": "1-2"
#     },
#     {
#         "char": "middle_evaluator_food_importance",
#         "initial_prompt": initial_importance_prompt_middle_bundle,
#         "consideration": "3"
#     },
#     {
#         "char": "good_evaluator_food_importance",
#         "initial_prompt": initial_importance_prompt_good_bundle,
#         "consideration": "4-5"
#     }
# ]

# # Loop through the configurations to run the refinement process
# for config in evaluator_configs:
#     # Run the version with metrics
#     await structure_refinement(
#         char = config["char"],
#         initial_prompt = config["initial_prompt"],
#         inferring_reasons = inferring_reasons,
#         refining_prompts = refining_prompts,
#         augmenting_prompts = augmenting_prompts,
#         training_data = food_train_data,
#         validation_data = food_val_data,
#         constant_metrics = adding_metrics,
#         metastructure = None,
#         mode = 'single',
#         consideration = config["consideration"],
#         convincing = convincing_prompt
#     )



# evaluator_configs = [
#     {
#         "char": "bad_evaluator_food_importance_no_experts",
#         "initial_prompt": initial_importance_prompt_bad_bundle,
#         "consideration": "1-2"
#     },
#     {
#         "char": "middle_evaluator_food_importance_no_experts",
#         "initial_prompt": initial_importance_prompt_middle_bundle,
#         "consideration": "3"
#     },
#     {
#         "char": "good_evaluator_food_importance_no_experts",
#         "initial_prompt": initial_importance_prompt_good_bundle,
#         "consideration": "4-5"
#     }
# ]

# # Loop through the configurations to run the refinement process
# for config in evaluator_configs:
#     # Run the version with metrics
#     await structure_refinement(
#         char = config["char"],
#         initial_prompt = config["initial_prompt"],
#         inferring_reasons = inferring_reasons_no_experts,
#         refining_prompts = refining_prompts,
#         augmenting_prompts = augmenting_prompts,
#         training_data = food_train_data,
#         validation_data = food_val_data,
#         constant_metrics = adding_metrics,
#         metastructure = None,
#         mode = 'single',
#         consideration = config["consideration"],
#         convincing = ""
#     )




